In [1]:
import pandas as pd
df = pd.read_csv("dataset.csv")

df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [ ]:
df= df.drop(['step','nameOrig','nameDest','isFlaggedFraud'])

In [4]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')

In [2]:
df['isFraud'].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [3]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [6]:


# Select required columns
df = df[['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
         'oldbalanceDest', 'newbalanceDest', 'isFraud']]

# Separate classes
fraud = df[df['isFraud'] == 1]
not_fraud = df[df['isFraud'] == 0]

# Undersample majority class
not_fraud_sampled = not_fraud.sample(n=len(fraud), random_state=42)

# Combine both classes
balanced_df = pd.concat([fraud, not_fraud_sampled])

# Shuffle dataset
balanced_df = balanced_df.sample(frac=1, random_state=42)

# Check new distribution
print(balanced_df['isFraud'].value_counts())

# Export to CSV
balanced_df.to_csv("balanced_fraud_dataset.csv", index=False)

print("Balanced dataset exported successfully!")

isFraud
0    8213
1    8213
Name: count, dtype: int64
Balanced dataset exported successfully!


In [7]:
data = pd.read_csv("balanced_fraud_dataset.csv")
data.head()

,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,CASH_IN,76550.74,1096252.93,1172803.66,2208784.02,2132233.28,0
1,PAYMENT,12617.11,339181.87,326564.76,0.00,0.00,0
2,CASH_OUT,8055.06,8055.06,0.00,0.00,8055.06,1
3,TRANSFER,342309.91,342309.91,0.00,0.00,0.00,1
4,CASH_OUT,2581549.92,2581549.92,0.00,0.00,2581549.92,1


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16426 entries, 0 to 16425
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   type            16426 non-null  object 
 1   amount          16426 non-null  float64
 2   oldbalanceOrg   16426 non-null  float64
 3   newbalanceOrig  16426 non-null  float64
 4   oldbalanceDest  16426 non-null  float64
 5   newbalanceDest  16426 non-null  float64
 6   isFraud         16426 non-null  int64  
dtypes: float64(5), int64(1), object(1)
memory usage: 898.4+ KB


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# # Load dataset
# data = pd.read_csv("balanced_fraud_dataset.csv")

# Features and target
X = data.drop("isFraud", axis=1)
y = data["isFraud"]

# Categorical and numerical columns
categorical_features = ["type"]
numerical_features = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

# Pipeline with RandomForest
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train model
pipeline.fit(X_train, y_train)

# Predictions
y_pred = pipeline.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9942178940961656

Confusion Matrix:
 [[1625   18]
 [   1 1642]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99      1643
           1       0.99      1.00      0.99      1643

    accuracy                           0.99      3286
   macro avg       0.99      0.99      0.99      3286
weighted avg       0.99      0.99      0.99      3286



In [15]:
print(data['type'].value_counts())

type
CASH_OUT    7066
TRANSFER    4750
PAYMENT     2735
CASH_IN     1825
DEBIT         50
Name: count, dtype: int64


In [10]:
import joblib

joblib.dump(pipeline, "fraud_detection_rf.joblib")

['fraud_detection_rf.joblib']

In [11]:
import joblib

model = joblib.load("fraud_detection_rf.joblib")

In [14]:
sample = {
    "type": ["CASH_OUT"],
    "amount": [8055],
    "oldbalanceOrg": [8055],
    "newbalanceOrig": [0],
    "oldbalanceDest": [0],
    "newbalanceDest": [8055]
}

import pandas as pd
sample_df = pd.DataFrame(sample)

prediction = model.predict(sample_df)

print("Fraud Prediction:", prediction)

Fraud Prediction: [1]
